# 🧠 AN-RA Training Notebook
**5 cells. Run top to bottom every session.**

| Cell | What it does |
|------|-------------|
| 1 | ⚙️ Config — only thing you ever edit |
| 2 | 🔧 Environment — GPU check + Drive mount + install deps |
| 3 | 📤 Repo — upload zip + apply fix + pre-flight check |
| 4 | 🚀 Run training |
| 5 | ✅ Verify Drive saves |


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║         AN-RA CONFIG — ONLY EDIT THIS CELL          ║
# ╚══════════════════════════════════════════════════════╝

SESSION_MINUTES    = 90    # ← change freely: 30, 60, 90, 110
OUROBOROS_MINUTES  = 10    # ← Ouroboros recursive reasoning
OVERHEAD_MINUTES   = 5     # ← reserved for setup/save

# Training mode:
#   "train"   → full pipeline (identity + ouroboros + RLVR) ✅ recommended
#   "session" → base transformer only (faster, lighter)
TRAINING_MODE = "train"

# ── Paths (do not edit) ───────────────────────────────────
REPO_PATH            = "/content/An-Ra-the-new-AGI"          # extracted location
DRIVE_ANRA_DIR       = "/content/drive/MyDrive/AnRa"
DRIVE_V2_CHECKPOINTS = "/content/drive/MyDrive/AnRa/v2/checkpoints"
DRIVE_DATASET        = "/content/drive/MyDrive/AnRa/anra_training.txt"   # canonical name
DRIVE_DATASET_LEGACY = "/content/drive/MyDrive/AnRa/anra_dataset_v6_1.txt"  # fallback

IDENTITY_MINUTES = max(1, SESSION_MINUTES - OVERHEAD_MINUTES - OUROBOROS_MINUTES)

print("╔══════════════════════════════════════════╗")
print("║         AN-RA SESSION PLAN               ║")
print("╠══════════════════════════════════════════╣")
print(f"║  Mode       : {TRAINING_MODE:<27}║")
print(f"║  Session    : {SESSION_MINUTES} min{' '*(25-len(str(SESSION_MINUTES)))}║")
print(f"║  Ouroboros  : {OUROBOROS_MINUTES} min{' '*(25-len(str(OUROBOROS_MINUTES)))}║")
print(f"║  Identity   : {IDENTITY_MINUTES} min (dynamic){' '*(14-len(str(IDENTITY_MINUTES)))}║")
print("╚══════════════════════════════════════════╝")


In [ ]:
# 🔧  ENVIRONMENT SETUP
import subprocess, sys, os, glob, time
import psutil, torch
from google.colab import drive

# ── 1. Hardware ───────────────────────────────────────────
print("── HARDWARE ────────────────────────────────────────")
try:
    raw = subprocess.check_output(
        ["nvidia-smi","--query-gpu=name,memory.total,memory.free,temperature.gpu",
         "--format=csv,noheader,nounits"], text=True
    ).strip().split(", ")
    print(f"  GPU   : {raw[0]}")
    print(f"  VRAM  : {int(raw[1])/1024:.1f} GB total | {int(raw[2])/1024:.1f} GB free | {raw[3]}°C")
except Exception:
    print("  GPU   : ❌ Not detected")

ram = psutil.virtual_memory()
print(f"  RAM   : {ram.available/1024**3:.1f} GB free / {ram.total/1024**3:.1f} GB total")
print(f"  Torch : {torch.__version__}  CUDA: {'✅' if torch.cuda.is_available() else '❌'}")

# ── 2. Mount Drive ────────────────────────────────────────
print("\n── GOOGLE DRIVE ────────────────────────────────────")
drive.mount("/content/drive")
for folder in [DRIVE_ANRA_DIR, DRIVE_V2_CHECKPOINTS,
               f"{DRIVE_ANRA_DIR}/identity", f"{DRIVE_ANRA_DIR}/logs",
               f"{DRIVE_ANRA_DIR}/v3/training"]:
    os.makedirs(folder, exist_ok=True)

pts = glob.glob(f"{DRIVE_V2_CHECKPOINTS}/*.pt")
dataset_ok = os.path.exists(DRIVE_DATASET) or os.path.exists(DRIVE_DATASET_LEGACY)
print(f"  Checkpoints : {len(pts)} {'— RESUME ✅' if pts else '— fresh start'}")
print(f"  Dataset     : {'✅ found' if dataset_ok else '❌ MISSING — upload anra_training.txt to Drive'}")

# ── 3. Install dependencies ───────────────────────────────
print("\n── DEPENDENCIES ────────────────────────────────────")
def pip(pkg):
    return subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True
    ).returncode == 0

CORE_PKGS = [
    "torch", "numpy", "sympy", "psutil", "tqdm",
    "transformers", "pytest", "sentencepiece",
    "faiss-cpu",      # memory vector search
    "gitpython",      # fs_agent git commits
    "networkx",       # knowledge graph
    "fastapi", "uvicorn",  # web server
]
for pkg in CORE_PKGS:
    ok = pip(pkg)
    print(f"  {'✅' if ok else '❌'} {pkg}")

MUON_AVAILABLE = False
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/KellerJordan/Muon"],
                   capture_output=True, timeout=30)
    import muon; MUON_AVAILABLE = True
    print("  ✅ muon")
except Exception:
    print("  ⚠️  muon unavailable — AdamW fallback")

print("\n✅ Environment ready")


In [ ]:
# 📤  REPO SETUP
from google.colab import files
import subprocess, zipfile, shutil, os, sys, glob, torch

# ── 1. Upload and extract ─────────────────────────────────
print("── UPLOAD ──────────────────────────────────────────")
print("Upload your An-Ra zip when the picker appears...")
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No file uploaded")

zip_name = list(uploaded.keys())[0]
if os.path.exists(REPO_PATH):
    shutil.rmtree(REPO_PATH)

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("/content/")

# Handle both standard GitHub zip extraction and already-normalized folders.
for candidate in [REPO_PATH + "-main", REPO_PATH]:
    if os.path.isdir(candidate) and candidate != REPO_PATH:
        os.rename(candidate, REPO_PATH)
        break

sys.path.insert(0, REPO_PATH)
print(f"  Repo: {'✅ ' + REPO_PATH if os.path.isdir(REPO_PATH) else '❌ not found at ' + REPO_PATH}")

# ── 2. Sync dataset from Drive ────────────────────────────
print("\n── DATASET ─────────────────────────────────────────")
local_data = f"{REPO_PATH}/training_data/anra_training.txt"
os.makedirs(f"{REPO_PATH}/training_data", exist_ok=True)

if os.path.exists(DRIVE_DATASET):
    shutil.copy2(DRIVE_DATASET, local_data)
    print(f"  ✅ Copied anra_training.txt ({os.path.getsize(local_data)/1024**2:.1f} MB)")
elif os.path.exists(DRIVE_DATASET_LEGACY):
    shutil.copy2(DRIVE_DATASET_LEGACY, local_data)
    print(f"  ✅ Copied legacy dataset as anra_training.txt ({os.path.getsize(local_data)/1024**2:.1f} MB)")
elif os.path.exists(local_data):
    print(f"  ✅ Dataset already in repo ({os.path.getsize(local_data)/1024**2:.1f} MB)")
else:
    print("  ⚠️  No dataset found — using bootstrap. Add data to Drive for better training.")

# ── 3. Sync checkpoints from Drive ───────────────────────
print("\n── CHECKPOINTS ─────────────────────────────────────")
ckpt_dir = f"{REPO_PATH}"
os.makedirs(ckpt_dir, exist_ok=True)
pts = glob.glob(f"{DRIVE_V2_CHECKPOINTS}/*.pt")
for pt in pts:
    dst = f"{ckpt_dir}/{os.path.basename(pt)}"
    if not os.path.exists(dst):
        shutil.copy2(pt, dst)
        print(f"  ✅ Restored {os.path.basename(pt)} ({os.path.getsize(dst)/1024**2:.1f} MB)")
    else:
        print(f"  ✅ Already present: {os.path.basename(pt)}")
RESUME_MODE = bool(pts)
print(f"  Resume mode: {'YES' if RESUME_MODE else 'NO — fresh start'}")

# ── 4. Validate tokenizer ─────────────────────────────────
print("\n── TOKENIZER ───────────────────────────────────────")
import json as _json
tok_path = f"{REPO_PATH}/tokenizer/tokenizer_v3.json"
if os.path.exists(tok_path):
    d = _json.load(open(tok_path))
    n = len(d.get("token_to_id", {}))
    if n == 8192:
        print(f"  ✅ tokenizer_v3.json: {n} entries")
    else:
        print(f"  ⚠️  tokenizer has {n} entries (expected 8192) — retraining...")
        result = subprocess.run(
            [sys.executable, "scripts/train_tokenizer_v3.py"],
            cwd=REPO_PATH, capture_output=True, text=True
        )
        print(result.stdout[-500:] if result.stdout else "")
        if result.returncode != 0:
            print(f"  ❌ Tokenizer training failed: {result.stderr[-300:]}")
        else:
            print("  ✅ Tokenizer retrained at 8192")
else:
    print("  ⚠️  tokenizer_v3.json missing — training...")
    subprocess.run([sys.executable, "scripts/train_tokenizer_v3.py"], cwd=REPO_PATH)

# ── 5. Pre-flight ─────────────────────────────────────────
print("\n── PRE-FLIGHT ──────────────────────────────────────")
checks = [
    (torch.cuda.is_available(),                    "CUDA available"),
    (os.path.isdir(REPO_PATH),                     "Repo extracted"),
    (os.path.exists(local_data),                   "Training data present"),
    (os.path.exists(DRIVE_V2_CHECKPOINTS),         "Drive checkpoint dir exists"),
    (os.path.exists(f"{REPO_PATH}/anra_paths.py"), "anra_paths.py present"),
    (os.path.exists(tok_path),                     "Tokenizer present"),
    (os.path.exists(f"{REPO_PATH}/identity/civ_watcher.py"), "civ_watcher.py present"),
]
checks.append((True, f"Resume mode: {'YES' if RESUME_MODE else 'NO — fresh'}"))

all_ok = True
for ok, label in checks:
    print(f"  {'✅' if ok else '❌'} {label}")
    if not ok: all_ok = False

if not all_ok:
    raise RuntimeError("❌ Pre-flight failed. Fix issues above before training.")
print("\n✅ All checks passed — ready to train")


In [ ]:
# 🚀  RUN TRAINING
import subprocess, sys, os, time

print("╔══════════════════════════════════════════╗")
print("║         AN-RA TRAINING STARTING          ║")
print("╠══════════════════════════════════════════╣")
print(f"║  Mode       : {TRAINING_MODE:<27}║")
print(f"║  Session    : {SESSION_MINUTES} min{' '*(25-len(str(SESSION_MINUTES)))}║")
print(f"║  Ouroboros  : {OUROBOROS_MINUTES} min{' '*(25-len(str(OUROBOROS_MINUTES)))}║")
print(f"║  Identity   : dynamic ({IDENTITY_MINUTES} min budget){' '*(13-len(str(IDENTITY_MINUTES)))}║")
print(f"║  Resume     : {'Yes' if RESUME_MODE else 'No (fresh)':<27}║")
print(f"║  Muon       : {'Yes' if MUON_AVAILABLE else 'No (AdamW fallback)':<27}║")
print("╚══════════════════════════════════════════╝\n")

cmd = [
    sys.executable, "-m", "training.train_unified",
    "--mode",              TRAINING_MODE,
    "--session_minutes",   str(SESSION_MINUTES),
    "--ouroboros_minutes", str(OUROBOROS_MINUTES),
]

env = os.environ.copy()
env["PYTHONPATH"] = REPO_PATH
start = time.time()

process = subprocess.Popen(cmd, cwd=REPO_PATH, env=env,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)
for line in process.stdout:
    print(line, end="", flush=True)
process.wait()

elapsed = (time.time() - start) / 60
print()
print("╔══════════════════════════════════════════╗")
if process.returncode == 0:
    print(f"║  ✅ Complete — {elapsed:.1f} min{'':>19}║")
else:
    print(f"║  ❌ Failed (code {process.returncode}) — {elapsed:.1f} min{'':>12}║")
print("╚══════════════════════════════════════════╝")
TRAINING_EXIT_CODE = process.returncode


In [ ]:
# ✅  VERIFY DRIVE SAVES
import os, glob

print("── DRIVE CHECKPOINT STATUS ──────────────────────")
pts   = glob.glob(f"{DRIVE_V2_CHECKPOINTS}/*.pt")
jsons = glob.glob(f"{DRIVE_V2_CHECKPOINTS}/*.json")

if pts or jsons:
    for f in sorted(pts + jsons):
        print(f"  ✅ {os.path.basename(f)} ({os.path.getsize(f)/1024**2:.1f} MB)")
else:
    print("  ⚠️  No checkpoint files found in Drive")
    print("  → Training may have failed — check Cell 4 output")

tok = f"/content/drive/MyDrive/AnRa/v2/tokenizer_v2.json"
print(f"\n  Tokenizer  : {'✅ on Drive' if os.path.exists(tok) else '⚠️  not yet saved'}")
print("\n🔁 Next session: run cells 1→4 again. Drive auto-restores.")


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║          CELL 8 — AN-RA SOVEREIGN UI                ║
# ║   Full chat interface with training controls        ║
# ╚══════════════════════════════════════════════════════╝
import subprocess, sys

# Install Flask
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flask", "flask-cors"],
               capture_output=True)

# Launch UI
REPO_PATH = open("/tmp/anra_repo_path.txt").read().strip()
exec(open(f"{REPO_PATH}/ui/anra_ui_launcher.py").read())

